In [47]:
# Load in libraries and match data

import json
import pandas as pd

with open("Single Match Data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

# Index to make sure we have correct order of events

df = df.sort_values("index").reset_index(drop=True)


In [48]:
df. head()

,id,index,period,timestamp,minute,second,possession,duration,type.id,type.name,...,shot.one_on_one,foul_committed.advantage,foul_won.advantage,clearance.aerial_won,pass.deflected,pass.no_touch,foul_committed.type.id,foul_committed.type.name,pass.straight,pass.goal_assist
0,9f6e2ecf-6685-45df-a62e-c2db3090f6c1,1,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0300039d-150d-41e4-b29a-76602ef002e6,2,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,491e8901-7630-4cc8-b57b-937dddff2eaa,3,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,757b85ad-ddfe-44d5-b893-c23a9fb709d8,4,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,5,1,00:00:00.575,0,0,2,2.015669,30,Pass,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
df["type.name"].value_counts()

type.name
Pass              1163
Ball Receipt*     1058
Carry              890
Pressure           212
Ball Recovery       89
Duel                53
Clearance           37
Goal Keeper         34
Block               32
Shot                28
Interception        24
Dribble             24
Foul Committed      23
Dispossessed        21
Foul Won            21
Miscontrol          17
Dribbled Past       14
Substitution         6
Half Start           4
Half End             4
Tactical Shift       4
Starting XI          2
Bad Behaviour        1
Error                1
Name: count, dtype: int64

In [50]:
# Don't want location like this [61.0, 40.1]

df["x"] = df["location"].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None
)

df["y"] = df["location"].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None
)

In [51]:
df["type.name"].value_counts()


type.name
Pass              1163
Ball Receipt*     1058
Carry              890
Pressure           212
Ball Recovery       89
Duel                53
Clearance           37
Goal Keeper         34
Block               32
Shot                28
Interception        24
Dribble             24
Foul Committed      23
Dispossessed        21
Foul Won            21
Miscontrol          17
Dribbled Past       14
Substitution         6
Half Start           4
Half End             4
Tactical Shift       4
Starting XI          2
Bad Behaviour        1
Error                1
Name: count, dtype: int64

In [52]:
df["possession"].nunique()


143

In [53]:
# See what shots look like for EDA

shots = df[df["type.name"] == "Shot"].copy()

shots[[
    "team.name",
    "player.name",
    "possession",
    "shot.statsbomb_xg",
    "shot.outcome.name"
]]

,team.name,player.name,possession,shot.statsbomb_xg,shot.outcome.name
136,Barcelona,Lionel Andrés Messi Cuccittini,6,0.076992,Off T
261,Barcelona,Jordi Alba Ramos,12,0.051668,Off T
714,Barcelona,Lionel Andrés Messi Cuccittini,23,0.016932,Saved
742,Deportivo Alavés,Rubén Sobrino Pozuelo,30,0.122604,Off T
801,Barcelona,Luis Alberto Suárez Díaz,33,0.041751,Off T
1340,Barcelona,Ousmane Dembélé,50,0.076063,Wayward
1546,Barcelona,Ivan Rakitić,58,0.112513,Off T
1587,Barcelona,Lionel Andrés Messi Cuccittini,61,0.049074,Post
1591,Barcelona,Gerard Piqué Bernabéu,61,0.115620,Off T
1619,Barcelona,Ousmane Dembélé,63,0.123355,Saved


In [ ]:
#Extract columns we want

cols = [
    "possession",
    "team.name",
    "player.name",
    "type.name",
    "x",
    "y",
    "shot.statsbomb_xg",
    "shot.outcome.name"
]

# Take out usesless values

df = df[
    ~df["type.name"].isin(["Half Start", "Starting XI"])
].copy()

# Half Start and Starting XI counted as possession 1 (which we have now taken out). We want to make the first true possession = 1 

df["possession"] = df["possession"] - 1


,possession,team.name,player.name,type.name,x,y,shot.statsbomb_xg,shot.outcome.name
4,1,Deportivo Alavés,Jonathan Rodríguez Menéndez,Pass,61.0,40.1,NaN,NaN
5,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Ball Receipt*,33.8,28.0,NaN,NaN
6,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Carry,33.8,28.0,NaN,NaN
7,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Pass,36.8,27.3,NaN,NaN
8,1,Deportivo Alavés,Rubén Sobrino Pozuelo,Ball Receipt*,79.8,75.9,NaN,NaN
...,...,...,...,...,...,...,...,...
79,4,Barcelona,Ivan Rakitić,Pass,69.6,12.8,NaN,NaN
80,4,Barcelona,Sergio Busquets i Burgos,Ball Receipt*,77.5,24.1,NaN,NaN
81,4,Barcelona,Sergio Busquets i Burgos,Carry,77.5,24.1,NaN,NaN
82,4,Barcelona,Sergio Busquets i Burgos,Pass,78.1,26.9,NaN,NaN


In [55]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(df[cols]).head(80)

,possession,team.name,player.name,type.name,x,y,shot.statsbomb_xg,shot.outcome.name
4,1,Deportivo Alavés,Jonathan Rodríguez Menéndez,Pass,61.0,40.1,NaN,NaN
5,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Ball Receipt*,33.8,28.0,NaN,NaN
6,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Carry,33.8,28.0,NaN,NaN
7,1,Deportivo Alavés,Guillermo Alfonso Maripán Loaysa,Pass,36.8,27.3,NaN,NaN
8,1,Deportivo Alavés,Rubén Sobrino Pozuelo,Ball Receipt*,79.8,75.9,NaN,NaN
9,1,Deportivo Alavés,Rubén Sobrino Pozuelo,Duel,86.5,74.2,NaN,NaN
10,2,Barcelona,Sergio Busquets i Burgos,Pass,33.6,5.9,NaN,NaN
11,2,Barcelona,Ivan Rakitić,Ball Receipt*,35.1,18.3,NaN,NaN
12,2,Barcelona,Ivan Rakitić,Pass,35.1,18.3,NaN,NaN
13,2,Barcelona,Ousmane Dembélé,Ball Receipt*,36.2,5.3,NaN,NaN


AttributeError: 'NoneType' object has no attribute 'head'